##### Importação dos Pacotes

In [0]:
import urllib.request
import zipfile
import geopandas as gpd
import os

##### Diretórios nos Catálogos

In [0]:
path_bronze_ibge_geo = "/Volumes/01_bronze/schemas/ibge/shp_geografico/"
path_silver_ibge_geo = "/Volumes/02_silver/schemas/ibge/"

##### Prearação para download dos arquivos (site: IBGE)

In [0]:
# Lista de UFs para downloads
uf = [
"AC","AL","AP","AM","BA","CE","DF","ES","GO",
"MA","MT","MS","MG","PA","PB","PR","PE","PI",
"RJ","RN","RS","RO","RR","SC","SP","SE","TO"
]

In [0]:
# Site do IBGE
# https://www.ibge.gov.br/geociencias/organizacao-do-territorio/malhas-territoriais.html

#base_url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2024/UFs"

##### Downloads dos arquivos no site do IBGE

In [0]:
## Loop para downloads dos arquivos
#for ufs in uf:

    ## Download de todos os Shapefiles do site do IBGE
    #url = f"{base_url}/{ufs}/{ufs}_Municipios_2024.zip"
    #destino = f"{path_bronze_ibge_geo}/{ufs}.zip"

    #urllib.request.urlretrieve(url, destino)

##### Descompactação dos arquivos do formato Zip

In [0]:
# Loop para descompactação dos arquivos
for ufs in uf:

    destino = f"{path_bronze_ibge_geo}/{ufs}.zip"
    
    # Descompactação dos arquivos do formato .zip
    with zipfile.ZipFile(destino, 'r') as zip_ref:
        zip_ref.extractall(path_bronze_ibge_geo)

In [0]:
# Lista todos os arquivos SHP usando Python
shp_files = [
    os.path.join(path_bronze_ibge_geo, f)
    for f in os.listdir(path_bronze_ibge_geo)
    if f.endswith(".shp")
]

##### Carrega arquivos (formato: Parquet) para camada: Silver

In [0]:
# Loop para salvar cada arquivo no formato Parquet na camada: Silver
for shp in shp_files:

    # Lê shapefile
    gdf = gpd.read_file(shp)

    # Geometry → String (Spark não entende geometry)
    gdf["geometry"] = gdf.geometry.astype(str)

    # Geopandas → Spark
    df_spark = spark.createDataFrame(gdf)

    # Nome do dataset
    nome = os.path.basename(shp).replace(".shp", "").lower()

    # Salva arquivo Parquet na camada: Silver
    df_spark.write.format("parquet") \
        .mode("overwrite") \
        .save(f"{path_silver_ibge_geo}/shp_geografico/{nome}")
        
    #print("Fim")

##### Deleta arquivos na camada: Bronze (exceto: *.zip)

In [0]:
# Lista todos os arquivos da pasta
files = dbutils.fs.ls(path_bronze_ibge_geo)

# Percorre e remove todos que NÃO são .zip
for file in files:
    if not file.path.lower().endswith(".zip"):
        dbutils.fs.rm(file.path, recurse=True)

##### Fim